In [1]:
import sys
sys.path.append("..")
import importlib
import clean_data

importlib.reload(clean_data)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path 
from catboost import CatBoostClassifier, Pool 

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, log_loss, f1_score, balanced_accuracy_score, accuracy_score

from clean_data import feature_engineering_days

In [2]:
df = pd.read_csv("../data/austin/raw/aac_intakes.csv")
outcomes = pd.read_csv("../data/austin/raw/aac_intakes_outcomes.csv")

df["datetime"] = pd.to_datetime(df["datetime"])
outcomes["intake_datetime"] = pd.to_datetime(outcomes["intake_datetime"])

df = df.merge(
    outcomes[["animal_id_intake", "intake_datetime", "time_in_shelter_days"]],
    left_on=["animal_id", "datetime"],
    right_on=["animal_id_intake", "intake_datetime"],
    how="inner"
)

df = df[df["animal_type"] == "Dog"].copy()

In [3]:
df = feature_engineering_days(df)

df_dogs = df[df["animal_type"] == "Dog"].copy()
df_dogs = df_dogs.sort_values("intake_datetime").reset_index(drop=True)

bins = [-np.inf, 7, 30, np.inf]
df_dogs["target"] = pd.cut(
    df_dogs["time_in_shelter_days"], bins=bins, labels=[0, 1, 2]
).astype(int)

In [ ]:
drop_cols = [
    "target", "time_in_shelter_days", "time_in_shelter", "intake_datetime", 
    "animal_type", "young_and_cat", "datetime"
]

X = df_dogs.drop(columns=drop_cols, errors="ignore")
y = df_dogs["target"]

#categorical features which will be available
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
X[cat_features] = X[cat_features].fillna("Unknown").astype(str)

#70/15/15
n = len(X)
i1, i2 = int(n * 0.70), int(n * 0.85)

X_train, X_val, X_test = X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:]
y_train, y_val, y_test = y.iloc[:i1], y.iloc[i1:i2], y.iloc[i2:]

print("Split:", len(X_train), len(X_val), len(X_test))

Split: 31434 6736 6737


In [5]:
model = CatBoostClassifier(
    iterations=1200, learning_rate=0.03, depth=6,
    loss_function="MultiClass", random_seed=42, verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    use_best_model=True
)

0:	learn: 1.0837698	test: 1.0818219	best: 1.0818219 (0)	total: 358ms	remaining: 7m 9s
100:	learn: 0.7958616	test: 0.7390389	best: 0.7390389 (100)	total: 20.7s	remaining: 3m 45s
200:	learn: 0.7739723	test: 0.7227043	best: 0.7227043 (200)	total: 36.6s	remaining: 3m 1s
300:	learn: 0.7621625	test: 0.7177162	best: 0.7177162 (300)	total: 53.8s	remaining: 2m 40s
400:	learn: 0.7534343	test: 0.7156548	best: 0.7156548 (400)	total: 1m 8s	remaining: 2m 16s
500:	learn: 0.7455541	test: 0.7135396	best: 0.7135396 (500)	total: 1m 19s	remaining: 1m 51s
600:	learn: 0.7395203	test: 0.7132542	best: 0.7131459 (545)	total: 1m 28s	remaining: 1m 28s
700:	learn: 0.7343070	test: 0.7126090	best: 0.7126090 (700)	total: 1m 36s	remaining: 1m 9s
800:	learn: 0.7289639	test: 0.7121292	best: 0.7121292 (800)	total: 1m 45s	remaining: 52.5s
900:	learn: 0.7243057	test: 0.7119287	best: 0.7118496 (863)	total: 1m 54s	remaining: 37.9s
1000:	learn: 0.7196335	test: 0.7115421	best: 0.7115151 (995)	total: 2m 2s	remaining: 24.4s
110

CatBoostClassifier(depth=6, iterations=1200, learning_rate=0.03, loss_function='MultiClass', random_seed=42, verbose=100)

In [6]:
X_train.columns

Index(['breed', 'color', 'intake_condition', 'intake_type', 'name', 'sex',
       'intact', 'is_mixed_breed', 'primary_breed', 'is_mixed_color',
       'primary_color', 'has_name', 'name_length', 'age_days', 'day_of_week',
       'month', 'year', 'is_weekend'],
      dtype='object')

# Evaluation


In [7]:
probs = model.predict_proba(X_test)
y_pred = model.classes_[np.argmax(probs, axis=1)]

#Log Loss
ll = log_loss(y_test, probs, labels=model.classes_)

#Multiclass Brier Score
y_onehot = (
    np.asarray(y_test)[:, None]
    == np.asarray(model.classes_)[None, :]
).astype(int)

brier = np.mean(np.sum((probs - y_onehot) ** 2, axis=1))

#Macro F1
f1 = f1_score(y_test, y_pred, average="macro")

#Balanced Accuracy
bal_acc = balanced_accuracy_score(y_test, y_pred)

print(f"Log Loss:          {ll:.4f}")
print(f"Brier Score:       {brier:.4f}")
print(f"Macro F1:          {f1:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred))

Log Loss:          0.7188
Brier Score:       0.4177
Macro F1:          0.5032
Balanced Accuracy: 0.4865

Classification report:
              precision    recall  f1-score   support

           0       0.75      0.90      0.82      4361
           1       0.50      0.33      0.40      1698
           2       0.41      0.23      0.29       678

    accuracy                           0.69      6737
   macro avg       0.55      0.49      0.50      6737
weighted avg       0.65      0.69      0.66      6737



In [8]:
from sklearn.metrics import precision_recall_curve, average_precision_score

classes = list(model.classes_)
i2 = classes.index(2)

p2_val = model.predict_proba(X_val)[:, i2]
p2_test = model.predict_proba(X_test)[:, i2]

# Wybór progu na walidacji: najwyższa precyzja przy recall >= 0.6
prec, rec, thr = precision_recall_curve((y_val == 2).astype(int), p2_val)
ok = rec[:-1] >= 0.8
threshold = thr[ok][np.argmax(prec[:-1][ok])]
print(f"Próg dla klasy >30 dni: {threshold:.3f}")

# Ocena na teście
flag = p2_test >= threshold
y_long = (y_test == 2).astype(int)
tp = np.sum(flag & (y_long == 1))
print(f"Oznaczone: {flag.mean():.1%} zwierząt")
print(f"Precyzja: {tp / max(flag.sum(), 1):.3f}")
print(f"Recall:   {tp / y_long.sum():.3f}")
print(f"PR-AUC (klasa >30): {average_precision_score(y_long, p2_test):.3f}")

Próg dla klasy >30 dni: 0.101
Oznaczone: 35.4% zwierząt
Precyzja: 0.209
Recall:   0.733
PR-AUC (klasa >30): 0.314


In [9]:
def predict_dog(model, dog):
    probs = model.predict_proba(dog)[0]

    return pd.DataFrame({
        "Czas pobytu": ["≤7 dni", "7–30 dni", ">30 dni"],
        "Prawdopodobieństwo (%)": [
            round(probs[list(model.classes_).index(i)] * 100, 2)
            for i in [0, 1, 2]
        ]
    })


In [10]:
dog = X_test.iloc[[0]]
print(df_dogs.loc[dog.index[0], "time_in_shelter_days"])
display(predict_dog(model, dog))

1.16875


,Czas pobytu,Prawdopodobieństwo (%)
0,≤7 dni,86.94
1,7–30 dni,11.59
2,>30 dni,1.47


In [11]:
dog = X_test.iloc[[30]]

print(df_dogs.loc[dog.index[0], "time_in_shelter_days"])

display(predict_dog(model, dog))

0.1284722222222222


,Czas pobytu,Prawdopodobieństwo (%)
0,≤7 dni,85.52
1,7–30 dni,12.71
2,>30 dni,1.77


In [12]:
long_stay = X_test[df_dogs.loc[X_test.index, "time_in_shelter_days"] > 60]

dog = long_stay.iloc[[0]]

print(df_dogs.loc[dog.index[0], "time_in_shelter_days"])
display(predict_dog(model, dog))


64.74027777777778


,Czas pobytu,Prawdopodobieństwo (%)
0,≤7 dni,43.45
1,7–30 dni,38.95
2,>30 dni,17.60


In [13]:
# import shap

# explainer = shap.TreeExplainer(model)
# X_sample = X_test.sample(n=min(500, len(X_test)), random_state=42)
# shap_values = explainer(X_sample)

# print(shap_values.values.shape)

In [14]:
# class_idx = list(model.classes_).index(2)
# shap_long = shap_values[:, :, class_idx]
# shap.plots.beeswarm(shap_long, max_display=15)

In [15]:
# dog = X_test.iloc[[0]]
# display(predict_dog(model, dog))

# dog_shap = explainer(dog)

# shap.plots.waterfall(
#     dog_shap[0, :, class_idx],
#     max_display=15
# )